# M2a contract v2 smoke run

This runs one audited full-size TLC month through the current v2 pipeline, then independently verifies the materialized tables. It is a pipeline smoke test, not the M2a fault benchmark.

Before running: attach the private source dataset created from the outer upload ZIP. Enable Internet and choose a CPU session. The source ZIP must be inside the outer ZIP together with its `.provenance.json` sidecar. The code below finds the input automatically.

The output archive omits the raw Parquet input. It records its SHA-256, and the local project already has the audited source files for later verification.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, tempfile, urllib.request, zipfile

INPUT_ROOT = Path('/kaggle/input')
SOURCE_NAME = 'data-integrity-compiler-source.zip'
source_candidates = list(INPUT_ROOT.rglob(SOURCE_NAME))
if len(source_candidates) != 1:
    raise RuntimeError(f'Expected one {SOURCE_NAME} under /kaggle/input, found {len(source_candidates)}. Check the attached dataset.')
source_zip = source_candidates[0]
sidecar = source_zip.with_suffix('.provenance.json')
if not sidecar.is_file():
    raise FileNotFoundError(f'Missing source provenance sidecar: {sidecar}')
provenance = json.loads(sidecar.read_text(encoding='utf-8'))
digest = hashlib.sha256()
with source_zip.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(block)
if digest.hexdigest() != provenance.get('source_archive_sha256'):
    raise ValueError('Uploaded source ZIP checksum does not match its sidecar')
with zipfile.ZipFile(source_zip) as archive:
    embedded_revision = json.loads(archive.read('SOURCE_REVISION.json'))
if any(embedded_revision.get(key) != provenance.get(key) for key in ('commit_sha', 'tree_sha')):
    raise ValueError('Embedded source revision and sidecar do not agree')
print('Verified source commit:', provenance['commit_sha'])
print('Verified source tree:', provenance['tree_sha'])


In [ ]:
root = Path(tempfile.mkdtemp(prefix='dic-m2a-v2-', dir='/kaggle/working')).resolve()
with zipfile.ZipFile(source_zip) as archive:
    for member in archive.infolist():
        destination = (root / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError(f'Unsafe path in source archive: {member.filename}')
    archive.extractall(root)
embedded = json.loads((root / 'SOURCE_REVISION.json').read_text(encoding='utf-8'))
if embedded != embedded_revision:
    raise ValueError('Extracted revision record differs from the checked archive record')

Path('/kaggle/temp').mkdir(parents=True, exist_ok=True)
venv = Path(tempfile.mkdtemp(prefix='dic-m2a-venv-', dir='/kaggle/temp'))
python = venv / 'bin' / 'python'
dic = venv / 'bin' / 'dic'
subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '-r', str(root / 'requirements-lock.txt')], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '-r', str(root / 'requirements-m2a-oracle-lock.txt')], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--no-deps', '-e', str(root)], check=True)
print('Project root:', root)
print('Isolated environment:', venv)


In [ ]:
raw = root / 'data' / 'raw'
raw.mkdir(parents=True, exist_ok=True)
trip_path = raw / 'yellow_tripdata_2023-01.parquet'
lookup_path = raw / 'taxi_zone_lookup.csv'
urllib.request.urlretrieve('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet', trip_path)
urllib.request.urlretrieve('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv', lookup_path)
print('Downloaded trip file bytes:', trip_path.stat().st_size)
print('Downloaded lookup bytes:', lookup_path.stat().st_size)

run = subprocess.run([str(dic), 'run', '--month', '2023-01', '--contract-version', 'v2'], cwd=root, text=True, capture_output=True)
if run.returncode != 0:
    raise RuntimeError(f'v2 run failed\nSTDOUT:\n{run.stdout}\nSTDERR:\n{run.stderr}')
report = json.loads(run.stdout)
print(json.dumps({
    'run_id': report['run_id'],
    'outcome': report['outcome'],
    'contract_hash': report['contract_hash'],
    'source_month': report['source_month'],
    'row_counts': report['row_counts'],
    'rejected_rows_by_rule': report['cleaning']['rejected_rows_by_rule'],
    'publication': report['publication'],
    'peak_memory': report['peak_memory'],
}, indent=2))

run_dir = root / 'data' / 'runs' / report['run_id']
verification_run = subprocess.run([
    str(python), '-m', 'experiments.m2a.verify_v2_run',
    '--root', str(root), '--run-dir', str(run_dir),
    '--source', str(trip_path), '--lookup', str(lookup_path),
], cwd=root, text=True, capture_output=True)
if verification_run.returncode != 0:
    raise RuntimeError(f'Independent verification failed\nSTDOUT:\n{verification_run.stdout}\nSTDERR:\n{verification_run.stderr}')
verification = json.loads(verification_run.stdout)
print(json.dumps(verification, indent=2))


In [ ]:
evidence = Path('/kaggle/working') / f'm2a-v2-smoke-{report["run_id"]}'
run_copy = evidence / 'data' / 'runs' / report['run_id']
run_copy.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, run_copy)

published = root / 'data' / 'published'
evidence_published = evidence / 'data' / 'published'
if (published / 'CURRENT').is_file():
    evidence_published.mkdir(parents=True, exist_ok=True)
    shutil.copy2(published / 'CURRENT', evidence_published / 'CURRENT')
    current_id = json.loads((published / 'CURRENT').read_text(encoding='utf-8'))['run_id']
    current_version = published / current_id
    current_copy = evidence_published / current_id
    current_copy.mkdir(parents=True, exist_ok=True)
    for name in ('manifest.json', 'report.json'):
        if (current_version / name).is_file():
            shutil.copy2(current_version / name, current_copy / name)

shutil.copytree(root / 'contracts' / 'v2', evidence / 'contracts' / 'v2')
audit_rel = json.loads((root / 'contracts' / 'v2' / 'pipeline.yaml').read_text())['source_audit']['path']
audit_dest = evidence / audit_rel
audit_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(root / audit_rel, audit_dest)
schema_dest = evidence / 'schemas'
schema_dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(root / 'schemas' / 'report-v2.schema.json', schema_dest / 'report-v2.schema.json')
(evidence / 'SOURCE_REVISION.json').write_text(json.dumps(embedded_revision, indent=2) + '\n', encoding='utf-8')
(evidence / 'verification.json').write_text(json.dumps(verification, indent=2) + '\n', encoding='utf-8')
metadata = {
    'run_id': report['run_id'], 'source_month': report['source_month'],
    'source_sha256': report['source_artifacts']['trips_sha256'],
    'lookup_sha256': report['source_artifacts']['lookup_sha256'],
    'contract_sha256': report['contract_hash'], 'outcome': report['outcome'],
    'verification': verification['verification'],
    'raw_source_included': False,
}
(evidence / 'evidence-summary.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
(evidence / 'README.txt').write_text(
    'M2a v2 one-month smoke evidence. Raw TLC source files are not included.\n'
    'Run the independent verifier from the project checkout, using this folder as --root,\n'
    'data/runs/<run_id> as --run-dir, and the matching local raw trip and lookup files.\n',
    encoding='utf-8',
)
archive_path = Path('/kaggle/working') / f'm2a-v2-smoke-{report["run_id"]}.zip'
shutil.make_archive(str(archive_path.with_suffix('')), 'zip', root_dir=evidence)
print('Evidence folder:', evidence)
print('Download this file from the Kaggle notebook Output panel:', archive_path)
print('Evidence ZIP bytes:', archive_path.stat().st_size)
